# Set up a new glacier (run once)

One-time onboarding for a new site, to run **before** the per-day
`4d_sfm` notebook. Three stages, in order:

1. **Standardize image names** - convert the raw camera tree to
   `<camera>_<YYYY-MM-DD>_<HHMMSS>.JPG` (automated; idempotent).
2. **MANUAL in Metashape** - build the GCP reference point cloud and
   calibrate the cameras, then save the `.psx`. No code here; this is GUI
   work you do by hand. The notebook waits at this stage.
3. **Bootstrap the reference registry** - read the `.psx`, export the
   calibrations, camera positions, and the dense **reference point cloud**
   (UTM `.laz`), and write `reference_registry.csv` (automated).

Each stage defines its own inputs in its own cell, so you fill in the
Stage 3 inputs only after the manual step is done. This notebook supersedes
the old `bootstrap_registry.ipynb` (Stage 3 is the same logic).

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"  # set before importing tlapse4d/Metashape

import pandas as pd
from pathlib import Path

from tlapse4d.preprocess import ensure_standardized
from tlapse4d.metashape import bootstrap_registry
import site_config_nogcp as site   # ← glacier paths (edit site_config.py)

## Stage 1 - Standardize image names

`ensure_standardized` looks at the filenames and only does work if needed:
if `raw_dir` already holds standard-named images it is returned as-is; if a
completed `<raw_dir>_renamed` already exists it is reused; otherwise the raw
tree is homogenised into a sibling `<raw_dir>_renamed` folder.

Note: the first run on a large raw tree reads EXIF from every image and
copies them, so it can take a while (hours for ~10k+ images). Re-running this
cell afterwards is instant. Requires ImageMagick (`identify`) on PATH, which
the `tlapse4d` conda env provides.

In [2]:
# Stage 1 input: the raw camera tree (one sub-folder per camera)
raw_dir = Path("/mnt/e/umayr/Changri/TLCAM/ChangriNorth_renamed")

tlcam_dir = ensure_standardized(raw_dir)
print(f"\ntlcam_dir = {tlcam_dir}")

Images already standard-named → using /mnt/e/umayr/Changri/TLCAM/ChangriNorth_renamed

tlcam_dir = /mnt/e/umayr/Changri/TLCAM/ChangriNorth_renamed


## Stage 2 - MANUAL: build the GCP reference cloud and calibrate cameras

Do this by hand in the Metashape GUI; there is no code for it. This produces
the reference `.psx` that Stage 3 reads. Adapt the steps to your exact GCP
workflow - the requirement is a single chunk in which:

1. The reference-day images are loaded - the drone/UAV survey images plus the
   time-lapse images for that day (the standardized ones from Stage 1).
2. Ground control points (markers) are placed and given their surveyed
   coordinates, then cameras are aligned and `Optimize Cameras` is run, so the
   chunk is properly georeferenced.
3. The dense reference point cloud / model is built (from the UAV survey +
   the reference-day time-lapse images) - Stage 3 exports this as the
   reference `.laz`.
4. Each time-lapse camera ends up as its own sensor with a refined
   calibration; the drone is a separate sensor.
5. Save the project as a `.psx` and note the exact **chunk label** shown in
   the workspace panel.

Then continue with Stage 3, filling in its inputs from what you just saved.

What Stage 3 then extracts automatically: per-camera calibration `.xml`s
(interior orientation), the camera position + orientation
(lon/lat/alt, yaw/pitch/roll - exterior orientation), and the dense
reference point cloud (UTM `.laz`). The drone images and sensor are skipped
automatically by filename shape.

## Stage 3 - Bootstrap the reference registry

Fill in the inputs below from the `.psx` you saved in Stage 2, then run.
Reads the reference `.psx`, exports the time-lapse cameras' calibration
(`.xml`) and position/orientation, reconstructs the per-camera image lists,
writes `reference_registry.csv`, **and exports the dense reference point
cloud as a UTM `.laz`** (returned as `result["ref_cloud"]` - pass that as
`ref_cloud` to the per-day pipeline). Idempotent: it skips itself if the
registry already exists (pass `overwrite=True` to rebuild).

In [3]:
# Stage 3 inputs - filled in AFTER the manual Metashape step (Stage 2)
ref_psx     = Path("/mnt/e/umayr/Changri/Changri_West/Sunday_Processing.psx")   # the .psx you saved
chunk_label = "WithoutGCP"        # exact chunk label from Metashape's workspace panel
ref_date    = "2023-11-27"    # the reference day (YYYY-MM-DD)

# Output paths come from site_config — the SAME source the monthly notebook
# reads, so setup and per-date processing can't disagree.
result = bootstrap_registry(
    ref_psx       = ref_psx,
    chunk_label   = chunk_label,
    ref_date      = ref_date,
    output_dir    = site.output_dir,
    registry_csv  = site.registry_csv,
    ref_cloud_out = site.ref_cloud,
    overwrite     = True,   # set True to rebuild from scratch
)
result

LoadProject: path = /mnt/e/umayr/Changri/Changri_West/Sunday_Processing.psx, read_only = on
loaded project in 0.039608 sec
Chunk   : 'WithoutGCP'  (365/365 aligned)
  Exported camera CSV : 2023-11-27_cameras_4DSfM.csv
  Skipping sensor : FC6310R (8.8mm)
  Exported calibration : C1.xml
  Exported calibration : C2.xml
  Exported calibration : C3.xml
  Exported calibration : C4.xml
  Exported calibration : C5.xml
  Registry updated : reference_registry.csv  (20 rows added, 20 total)
ExportPointCloud: path = /mnt/e/umayr/Changri/Changri_West_noGCP/output/Reference_UAV_TLC_PCS.laz, format = PointCloudFormatLAZ, crs = WGS 84 / UTM zone 45N
point cloud size: 125222166 points
saved 125222166 points
  Exported reference cloud : Reference_UAV_TLC_PCS.laz  (EPSG:32645, .laz)


{'registry_csv': PosixPath('/mnt/e/umayr/Changri/Changri_West_noGCP/output/reference_registry.csv'),
 'cameras_csv': PosixPath('/mnt/e/umayr/Changri/Changri_West_noGCP/output/2023-11-27/4D_SfM/2023-11-27_cameras_4DSfM.csv'),
 'calib_dir': PosixPath('/mnt/e/umayr/Changri/Changri_West_noGCP/output/2023-11-27/4D_SfM/adjusted_calib_4DSfM'),
 'ref_cloud': PosixPath('/mnt/e/umayr/Changri/Changri_West_noGCP/output/Reference_UAV_TLC_PCS.laz'),
 'n_cameras': 5,
 'n_sensors': 5,
 'cameras': ['C1', 'C2', 'C3', 'C4', 'C5']}

## Verify

Sanity-check the registry, then move to the `4d_sfm` notebook using the
`tlcam_dir` (Stage 1), `registry_csv`, and the exported reference cloud
`result["ref_cloud"]` (Stage 3) set above.

In [4]:
df_reg = pd.read_csv(site.registry_csv)
print(f"Registry rows      : {len(df_reg)}")
print(f"Dates in registry  : {df_reg['date'].unique().tolist()}")
print(f"Cameras in registry: {df_reg['label'].str.split('_').str[0].unique().tolist()}")
print(f"Calib dir          : {df_reg['calib_dir'].iloc[0]}")
print(f"Reference cloud    : {result['ref_cloud']}")
print("\nThe monthly notebook reads the same site_config — nothing to re-type.")
df_reg.head(10)

Registry rows      : 20
Dates in registry  : ['2023-11-27']
Cameras in registry: ['C1', 'C2', 'C3', 'C4', 'C5']
Calib dir          : /mnt/e/umayr/Changri/Changri_West_noGCP/output/2023-11-27/4D_SfM/adjusted_calib_4DSfM
Reference cloud    : /mnt/e/umayr/Changri/Changri_West_noGCP/output/Reference_UAV_TLC_PCS.laz

The monthly notebook reads the same site_config — nothing to re-type.


,date,label,image_path,lon,lat,alt,yaw,pitch,roll,calib_dir
0,2023-11-27,C1_2023-11-27_090001,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.771375,27.983199,5490.370306,221.677722,87.761329,57.602583,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
1,2023-11-27,C1_2023-11-27_100000,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.771375,27.983199,5490.342264,221.385025,87.753992,57.313057,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
2,2023-11-27,C1_2023-11-27_110001,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.771376,27.983197,5490.334629,221.930171,87.750896,57.866362,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
3,2023-11-27,C1_2023-11-27_120000,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.771375,27.983200,5490.360751,221.946963,87.754269,57.911149,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
4,2023-11-27,C2_2023-11-27_090001,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.772437,27.985370,5560.247745,298.613121,79.384803,125.707945,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
5,2023-11-27,C2_2023-11-27_100000,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.772438,27.985369,5560.375137,298.532874,79.397065,125.619783,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
6,2023-11-27,C2_2023-11-27_110000,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.772437,27.985370,5560.406672,298.424319,79.416193,125.557756,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
7,2023-11-27,C2_2023-11-27_120000,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.772436,27.985371,5560.401117,298.362098,79.425930,125.516661,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
8,2023-11-27,C3_2023-11-27_090001,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.775938,27.985941,5539.526434,224.062507,82.633473,29.987103,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
9,2023-11-27,C3_2023-11-27_100000,/mnt/e/umayr/Changri/TLCAM/ChangriWest_renamed...,86.775938,27.985940,5539.514712,224.113157,82.632772,30.048460,/mnt/e/umayr/Changri/Changri_West_noGCP/output...
